In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# ==========================================================
# 1. Data Loading and Preprocessing
# ==========================================================
# Load the liver cirrhosis dataset
df = pd.read_csv('liver_cirrhosis.csv')

# Select target blood biomarkers and drop missing values
biomarkers = df[['Albumin', 'Cholesterol', 'Tryglicerides']].dropna()

# ==========================================================
# 2. Physiological Sweat Data Simulation
# ==========================================================
# Set random seed for reproducibility
np.random.seed(42)

def sweat_blood_data(row):
    """
    Simulates sweat biomarker levels based on blood biomarker values.
    
    """
    # Calculate weight factors based on blood test results
    # When albumin level drops, and Cholesterol and Tryglicerides leveles increase, liver has issues.
    # Normal range of Albumin (3.5~5.5g/dL), 4.5g/dL is threshold
    albu_level_effect = max(0, (4.5 - row['Albumin']) * 5)
    # Normal range of Cholesterol (150~200mg/dL), 150mg/dL is threshold
    chol_level_effect = max(0, (row['Cholesterol'] - 150) / 50)
    # Normal range of Tryglicerides (<150mg/dL), 100mg/dL is threshold
    tri_level_effect = max(0, (row['Tryglicerides'] - 100) / 50) 
    
    combined_factor = albu_level_effect + chol_level_effect + tri_level_effect

    # Generate sweat data following distributions from Baker et al. (2022)
    return pd.Series({
        # Baseline mean: 3.0 mmol/L (Normal range: 1-10 mmol/L)
        'Ammonia': np.random.normal(3.0 + combined_factor, 1.0), 
        # Baseline mean: 15.0 mmol/L (Normal range: 10-50 mmol/L)
        'Urea': np.random.normal(15.0 + combined_factor * 2, 5.0), 
        # Baseline mean: 40.0 mmol/L (Normal range: 15-90 mmol/L)
        'Sodium': np.random.normal(40.0 + combined_factor * 3, 10.0), 
        # Baseline mean: 35.0 mmol/L (Normal range: 10-90 mmol/L)
        'Chloride': np.random.normal(35.0 + combined_factor * 2.5, 10.0),
        # Baseline mean: 4.0 mmol/L (Normal range: 3-10 mmol/L)
        'Potassium': np.random.normal(4.0 + combined_factor * 0.5, 1.0),
        # Baseline mean: 50.0 nmol/L (Normal range: 8-140 nmol/L)
        'Cortisol': np.random.normal(50.0 + combined_factor * 10, 20.0)
    })

# Apply simulation and merge datasets
sweat_features = biomarkers.apply(sweat_blood_data, axis=1)
final_df = pd.concat([biomarkers.reset_index(drop=True), 
                      sweat_features.reset_index(drop=True)], axis=1)

# ==========================================================
# 3. Model Training (Random Forest)
# ==========================================================
# Features (X): 6 sweat features / Targets (y): 3 blood biomarkers
X = final_df[['Ammonia', 'Urea', 'Sodium', 'Chloride', 'Potassium', 'Cortisol']]
y = final_df[['Albumin', 'Cholesterol', 'Tryglicerides']]

# Split dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train multi-output Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# ==========================================================
# 4. Model Prediction and Evaluation
# ==========================================================
y_pred = rf_model.predict(X_test)

print("=== [System Performance Evaluation] ===")
print(f"Total R2 Score (Accuracy): {r2_score(y_test, y_pred):.4f}")
print(f"Total MAE (Mean Absolute Error): {mean_absolute_error(y_test, y_pred):.4f}\n")

# Detailed metrics for each blood biomarker
target_names = ['Albumin', 'Cholesterol', 'Tryglicerides']

for i, name in enumerate(target_names):
    r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])
    mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
    print(f"[{name}] R2: {r2:.4f}, MAE: {mae:.4f}")

=== [System Performance Evaluation] ===
Total R2 Score (Accuracy): 0.5290
Total MAE (Mean Absolute Error): 26.7108

[Albumin] R2: 0.4067, MAE: 0.2216
[Cholesterol] R2: 0.8771, MAE: 51.0212
[Tryglicerides] R2: 0.3031, MAE: 28.8895


In [2]:
# ==========================================================
# 5. System Validation: Real-Time Inference Simulation
# ==========================================================
"""
Note: The actual system is designed for automated real-time streaming.
This manual input module is for model validation purposes only.
"""
    
def liver_health_notice(predictions):
    """
    Classifies the health risk level based on predicted blood biomarkers.
    Logic: Priority-based classification of hepatic and metabolic risks.
    """
    alb, chol, tri = predictions
    
    # 1. Critical: Combined risk of liver and metabolic dysfunction
    if alb < 3.5 and (chol >= 240 or tri >= 200):
        return "🚨 [Critical] High Risk: Combined liver & metabolic issues. Urgent medical consultation required."
    
    # 2. Danger: Hepatic dysfunction only (Low Albumin)
    elif alb < 3.5:
        return "🔴 [Danger] Liver Health: Low Albumin detected. Suspected hepatic impairment."
    
    # 3. Warning: Severe lipid levels
    elif chol >= 240 and tri >= 200:
        return "🟡 [Warning] High Lipids: Severe cholesterol and triglyceride levels detected. Diet management required."
    
    # 4. Notice: Single Metabolic Risk (Either Cholesterol or Triglycerides is high)
    elif chol >= 240 or tri >= 200:
        return "🟠 [Notice] Metabolism: High cholesterol or fat. Regular health monitoring advised."
    
    # 5. Healthy: All indicators within normal range
    else:
        return "✅ [Normal] All healthy! Maintain your current lifestyle."

def trend_analysis(current_pred):
    """
    Compares current results with precise clinical ranges:
    - Albumin: 3.5 to 5.5 g/dL (Cleveland Clinic)
    - Cholesterol: < 200 mg/dL
    - Triglycerides: < 150 mg/dL
    """
    # 1. Historical Data
    monthly_avg = [3.80, 255.00, 115.00] 
    
    # 2. Clinical Reference Standards
    labels = ['Albumin', 'Cholesterol', 'Triglycerides']
    
    print("\n" + "="*60)
    print("   [ CLINICAL REFERENCE & TREND REPORT ]")
    print("="*60)

    # PART 1: Clinical Benchmarking (Normal Range-Based)
    print("--- Clinical Reference Comparison (Standard Range) ---")
    for i, name in enumerate(labels):
        val = current_pred[i]
        
        if name == 'Albumin':
            # Albumin Range Check: 3.5 <= val <= 5.5
            is_normal = (val >= 3.5) and (val <= 5.5)
            ref_text = "(Normal: 3.5 - 5.5)"
        elif name == 'Cholesterol':
            is_normal = val < 200
            ref_text = "(Normal: < 200)"
        else: # Triglycerides
            is_normal = val < 150
            ref_text = "(Normal: < 150)"
        
        status = "Normal ✅" if is_normal else "Out of Range ⚠️"
        print(f" > {name:<13}: {val:.2f} {ref_text:<18} | {status}")
    print("-" * 60)

    # PART 2: Personal Trend Analysis (Same as before)
    print(f"--- Comparison with Monthly Average ---")
    for i, name in enumerate(labels):
        diff = current_pred[i] - monthly_avg[i]
        if name == 'Albumin':
            trend = "Improved ✅" if diff > 0 else "Worsened ⚠️"
        else:
            trend = "Improved ✅" if diff < 0 else "Worsened ⚠️"
        print(f" > {name:<13}: {current_pred[i]:.2f} (vs {monthly_avg[i]:.2f}) | {trend} ({diff:+.2f})")
    print("-" * 60)
    
def run_interactive_diagnosis():
    """
    Allows the user to enter 6 sweat values and 
    performs real-time inference using the trained Random Forest model.
    """
    print("\n" + "="*60)
    print("   [ INTERACTIVE REAL-TIME SWEAT ANALYSIS SYSTEM ]")
    print("="*60)
    print("Please input the measured sweat analyte levels for clinical inference:")

    try:
        # User input for 6 sweat analytes (Features)
        val1 = float(input(" 1. Ammonia (mmol/L)   : "))
        val2 = float(input(" 2. Urea (mmol/L)      : "))
        val3 = float(input(" 3. Sodium (mmol/L)    : "))
        val4 = float(input(" 4. Chloride (mmol/L)  : "))
        val5 = float(input(" 5. Potassium (mmol/L) : "))
        val6 = float(input(" 6. Cortisol (nmol/L)  : "))

        # Formatting input into a DataFrame for model compatibility
        new_data = pd.DataFrame([[val1, val2, val3, val4, val5, val6]], 
                                columns=['Ammonia', 'Urea', 'Sodium', 'Chloride', 'Potassium', 'Cortisol'])

        # AI-driven Inference (Predicting Blood Biomarkers)
        new_pred = rf_model.predict(new_data)[0]

        print("\n" + "-"*60)
        print(" [ AI INFERENCE RESULT: PREDICTED BLOOD LEVELS ]")
        print(f" > Albumin      : {new_pred[0]:.2f} g/dL")
        print(f" > Cholesterol  : {new_pred[1]:.2f} mg/dL")
        print(f" > Triglycerides: {new_pred[2]:.2f} mg/dL")
        print("-" * 60)
        
        # Displaying the final clinical diagnosis
        print(" [ CLINICAL DIAGNOSIS & ACTION NOTICE ]")
        print(f" {liver_health_notice(new_pred)}")
        
        # Output: Trend Analysis
        trend_analysis(new_pred)
        
        print("="*60 + "\n")

    except ValueError:
        print("\n[Error] Invalid input. Please ensure all inputs are numeric values.")

# Execute the interactive module
run_interactive_diagnosis()


   [ INTERACTIVE REAL-TIME SWEAT ANALYSIS SYSTEM ]
Please input the measured sweat analyte levels for clinical inference:
 1. Ammonia (mmol/L)   : 50
 2. Urea (mmol/L)      : 50
 3. Sodium (mmol/L)    : 50
 4. Chloride (mmol/L)  : 50
 5. Potassium (mmol/L) : 50
 6. Cortisol (nmol/L)  : 50

------------------------------------------------------------
 [ AI INFERENCE RESULT: PREDICTED BLOOD LEVELS ]
 > Albumin      : 3.13 g/dL
 > Cholesterol  : 1586.74 mg/dL
 > Triglycerides: 174.43 mg/dL
------------------------------------------------------------
 [ CLINICAL DIAGNOSIS & ACTION NOTICE ]
 🚨 [Critical] High Risk: Combined liver & metabolic issues. Urgent medical consultation required.

   [ CLINICAL REFERENCE & TREND REPORT ]
--- Clinical Reference Comparison (Standard Range) ---
 > Albumin      : 3.13 (Normal: 3.5 - 5.5) | Out of Range ⚠️
 > Cholesterol  : 1586.74 (Normal: < 200)    | Out of Range ⚠️
 > Triglycerides: 174.43 (Normal: < 150)    | Out of Range ⚠️
-------------------------

In [3]:
# ==========================================================
# 6. Dynamic Stream Processing: Sequential Biometric Inference
# ==========================================================

def run_realtime_diagnostic(new_sweat_data):
    """
    Processes a single data packet received from a virtual sensor.
    Performs inference, generates a clinical notice, and analyzes trends.
    """
    # Convert raw dictionary input into a DataFrame for the model
    input_df = pd.DataFrame([new_sweat_data])

    # Perform model inference to predict blood biomarker levels
    # Extract the first row of the output
    predicted_blood_values = rf_model.predict(input_df)[0]
    
    # Print the real-time analysis results
    print("\n" + ">>>" * 20)
    print(" [ INCOMING DATA PACKET PROCESSED ]")
    print(f" Received Sensor Values: {new_sweat_data}")
    print("-" * 60)
    
    # Generate and display the Clinical Diagnosis (Notice)
    clinical_notice = liver_health_notice(predicted_blood_values)
    print(f" AI Clinical Diagnosis -> {clinical_notice}")
    
    # Perform Longitudinal Trend Analysis (Comparing with History)
    trend_analysis(predicted_blood_values)
    
    print(">>>" * 20 + "\n")

# --- Simulation: Automated Data Streaming ---
# These packets simulate continuous data received from a wearable device
incoming_stream = [
    {'Ammonia': 4.5, 'Urea': 25.0, 'Sodium': 50.0, 'Chloride': 45.0, 'Potassium': 4.5, 'Cortisol': 70.0},
    {'Ammonia': 12.0, 'Urea': 60.0, 'Sodium': 80.0, 'Chloride': 75.0, 'Potassium': 6.0, 'Cortisol': 150.0}, # Abnormal case
    {'Ammonia': 2.8, 'Urea': 14.5, 'Sodium': 38.0, 'Chloride': 33.0, 'Potassium': 4.1, 'Cortisol': 45.0}    # Healthy case
]

# Execute the pipeline for each incoming data point
print("Starting Real-time Biometric Monitoring Pipeline...")
for data_point in incoming_stream:
    run_realtime_diagnostic(data_point)

Starting Real-time Biometric Monitoring Pipeline...

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
 [ INCOMING DATA PACKET PROCESSED ]
 Received Sensor Values: {'Ammonia': 4.5, 'Urea': 25.0, 'Sodium': 50.0, 'Chloride': 45.0, 'Potassium': 4.5, 'Cortisol': 70.0}
------------------------------------------------------------
 AI Clinical Diagnosis -> ✅ [Normal] All healthy! Maintain your current lifestyle.

   [ CLINICAL REFERENCE & TREND REPORT ]
--- Clinical Reference Comparison (Standard Range) ---
 > Albumin      : 4.13 (Normal: 3.5 - 5.5) | Normal ✅
 > Cholesterol  : 199.77 (Normal: < 200)    | Normal ✅
 > Triglycerides: 68.47 (Normal: < 150)    | Normal ✅
------------------------------------------------------------
--- Comparison with Monthly Average ---
 > Albumin      : 4.13 (vs 3.80) | Improved ✅ (+0.33)
 > Cholesterol  : 199.77 (vs 255.00) | Improved ✅ (-55.23)
 > Triglycerides: 68.47 (vs 115.00) | Improved ✅ (-46.53)
------------------------------------------------